# Brain Tumor Segmentation *BraTS&mdash;Style*
This notebook will implement a *BraTS-Style* Lighting pipeline that will train a model to preform brain tumor segmentation.  The trained model will then be used to segment ***post-operative*** brain tumor MRI data.  Below is the data directory structure that will be used for this study.
```
BraTS/
├── MICCAI-BraTS-2021/
│   ├── train/      # BraTS training data (T1, T1ce, T2, FLAIR + mask)
│   └── postop/     # Post-op data (same modalities, no mask)
```
- Data loading, transformations, and model training metrix collections will be accomplished using the **MONAI** package. ***MONAI*** is optimized for MRI-specific transforms, loss functions, and evaluation metrics for medical imaging
- A **PyTroch/Lighting** class is wrapped around a pre-trained UNet model that is used to produce the BraTS segmentation lables which are defined as:
    - 1 = NC/Necrosis
    - 2 = Edema
    - 4 = Enhancing tumor
- **Transfer Learning** the labels that are learned during training are then transferred to the **post-operative** images for segmentation

----
<a name='startup_tasks'></a>
## 1.0 <span style='color:blue'>|</span> Common Start Up Tasks

<a name='load_packages'></a>
### 1.1 <span style='color:blue'>|</span> Load Required Libraries and Packages

In [1]:
import os, glob
import numpy as np
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' # Fixes a warning from Pytorch
import torch, monai
import torch.nn as nn
import torchio as tio
import torch.nn.functional as F
import pytorch_lightning as pl

from torchmetrics.segmentation import DiceScore
from torch.utils.data import DataLoader
from pytorch_lightning import LightningDataModule
import nibabel as nib
from pytorch3dunet.unet3d.model import UNet3D

# Squash Python warnings
import warnings
warnings.filterwarnings('ignore')

# Enable Python's Garbage Collector
import gc
gc.collect()

<frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-02-20 16:45:59.743309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


<a name='global_variables'></a>
### 1.2 <span style='color:blue'>|</span> Declare Global Variables

In [5]:
DATA_DIR = '../BraTS/'
TRN_DIR = DATA_DIR+'BraTS2021_TrainingSet/UPENN-GBM/'
VAL_DIR = DATA_DIR+'BraTS2021_ValidationSet/UPENN-GBM/'
POST_DIR = '../post'

In [2]:
class DataModule(LightningDataModule):
    def __init__(self, train_dir, val_dir=None, batch_size=2, num_workers=4, patch_size=(128, 128, 128)):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.patch_size = patch_size

        # Preprocessing (CPU/GPU compatible)
        self.preprocessing = tio.Compose([
            tio.Resample((1.0, 1.0, 1.0)),
            tio.ToCanonical(),
            tio.CropOrPad(self.patch_size, padding_mode=0),
            tio.ZNormalization(masking_method=tio.ZNormalization.MEAN),
        ])

    def setup(self, stage=None):
        # Load patients
        subjects_train = self._load_subjects(self.train_dir, stage='train')
        self.train_ds = tio.SubjectsDataset(subjects_train, transform=self.preprocessing)

        if self.val_dir:
            subjects_val = self._load_subjects(self.val_dir, stage='val')
            self.val_ds = tio.SubjectsDataset(subjects_val, transform=self.preprocessing)

    def _load_subjects(self, directory, stage='train'):
        subjects = []
        for patient_dir in sorted(os.listdir(directory)):
            patient_path = os.path.join(directory, patient_dir)
            if not os.path.isdir(patient_path):
                continue

            # Expect 4 modalities: T1, T1ce, T2, FLAIR (case-insensitive)
            flair = self._find_modality(patient_path, ['flair'])
            t1 = self._find_modality(patient_path, ['t1', 't1n', 't1-weighted'])
            t1ce = self._find_modality(patient_path, ['t1ce', 't1c', 't1-gad'])
            t2 = self._find_modality(patient_path, ['t2', 't2-weighted'])

            if not all([flair, t1, t1ce, t2]):
                continue

            # Load label (BraTS format: 0,1,2,4 → remapped to [0,1,2,3])
            label_path = self._find_label(patient_path)
            if stage != 'predict' and not label_path:
                continue  # skip patients without labels in train/val

            # Create subject
            subject_data = {
                'flair': tio.Image(flair, type=tio.INTENSITY),
                't1': tio.Image(t1, type=tio.INTENSITY),
                't1ce': tio.Image(t1ce, type=tio.INTENSITY),
                't2': tio.Image(t2, type=tio.INTENSITY),
            }

            if label_path:
                label_img = nib.load(label_path).get_fdata()
                # Remap: 1/4→1, 2→2, 0→0, 3→1 (as per BraTS)
                label_map = np.zeros_like(label_img, dtype=np.uint8)
                label_map[label_img == 1] = 1  # necrosis
                label_map[label_img == 2] = 2  # edema
                label_map[np.isin(label_img, [3, 4])] = 1  # enhancing & active tumor
                label_img = torch.from_numpy(label_map)

                subject_data['label'] = tio.LabelMap(tensor=label_img, type=tio.LABEL)

            subject = tio.Subject(subject_data)
            subjects.append(subject)

        return subjects

    def _find_modality(self, path, patterns):
        for f in os.listdir(path):
            if any(p in f.lower() for p in patterns):
                return os.path.join(path, f)
        return None

    def _find_label(self, path):
        for f in os.listdir(path):
            if 'seg' in f.lower():
                return os.path.join(path, f)
        return None

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

In [3]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_filters=32):
        super().__init__()

        # Encoder
        self.enc1 = ConvBlock(in_channels, base_filters)
        self.pool = nn.MaxPool3d(2)

        self.enc2 = ConvBlock(base_filters, base_filters * 2)
        self.enc3 = ConvBlock(base_filters * 2, base_filters * 4)
        self.enc4 = ConvBlock(base_filters * 4, base_filters * 8)

        # Bottleneck
        self.bottleneck = ConvBlock(base_filters * 8, base_filters * 16)

        # Decoder
        self.upconv4 = nn.ConvTranspose3d(base_filters * 16, base_filters * 8, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(base_filters * 16, base_filters * 8)

        self.upconv3 = nn.ConvTranspose3d(base_filters * 8, base_filters * 4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(base_filters * 8, base_filters * 4)

        self.upconv2 = nn.ConvTranspose3d(base_filters * 4, base_filters * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base_filters * 4, base_filters * 2)

        self.upconv1 = nn.ConvTranspose3d(base_filters * 2, base_filters, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base_filters * 2, base_filters)

        # Final output
        self.final_conv = nn.Conv3d(base_filters, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))

        bottleneck = self.bottleneck(self.pool(enc4))

        # Decoder with skip connections
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)

        return self.final_conv(dec1)

In [4]:
class LightningModule(pl.LightningModule):
    def __init__(self, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = UNet3D(in_channels=4, out_channels=3)
        self.dice = Dice(num_classes=3, average='macro', mdmc_average='global')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def _shared_step(self, batch, stage):
        image, label = batch['image'], batch['label']  # image: [B,4,H,W,D], label: [B,3,H,W,D]
        pred = self(image)
        loss = F.cross_entropy(pred, torch.argmax(label, dim=1))  # label is one-hot
        dice = self.dice(pred.softmax(dim=1), label.argmax(dim=1))

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_dice', dice, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return {'optimizer': optimizer}

In [ ]:
def main():
    # Paths (adjust to your setup)
    train_dir = '/path/to/BraTS20/train'
    val_dir = '/path/to/BraTS20/val'

    data_module = DataModule(
        train_dir=train_dir, val_dir=val_dir,
        batch_size=2, num_workers=4,
        patch_size=(128, 128, 128)
    )

    model = LightningModule(learning_rate=1e-4)

    trainer = pl.Trainer(
        max_epochs=100,
        accelerator='gpu',
        devices=1,
        logger=True,
        log_every_n_steps=10,
        enable_checkpointing=True,
        precision=16,  # mixed precision for faster training
    )

    trainer.fit(model, data_module)

if __name__ == '__main__':
    main()